# XAUUSD M5 Scalping ML v2
### 9 Fixes: ATR target Ã‚Â· Market Structure Ã‚Â· Liquidity Ã‚Â· Candle Patterns Ã‚Â· Area Context Ã‚Â· Hybrid Backtest Ã‚Â· Session Filter Ã‚Â· Dynamic Threshold Ã‚Â· No-trade Zone

In [1]:
import subprocess, sys
for pkg in ["lightgbm","xgboost","scikit-learn","optuna","shap"]:
    try: __import__(pkg.replace("-","_"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
print("Library siap")


Library siap


## Config + Imports

In [2]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings, joblib, json, os
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, precision_score, f1_score
from sklearn.calibration import CalibratedClassifierCV
import lightgbm as lgb, xgboost as xgb, optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

# ── CONFIG ─────────────────────────────────────────────
SYMBOL      = "XAUUSD"
LOT         = 0.01
SPREAD_USD  = 0.25
LOOKAHEAD   = 20     # 20 candle M5 = 100 menit — cukup waktu untuk TP/SL kena
FIXED_SL    = 6.0    # pip — harus sama dengan FIXED_SL_PIPS di config.py
FIXED_TP    = 8.0    # pip — harus sama dengan FIXED_TP_PIPS di config.py
ATR_TP_MULT = FIXED_TP / 6.0   # referensi saja (ATR M5 XAUUSD ~6)
ATR_SL_MULT = FIXED_SL / 6.0
PROB_LOW_VOL  = 0.52  # threshold sinyal volatilitas rendah
PROB_HIGH_VOL = 0.57  # threshold sinyal volatilitas tinggi (ATR >= ATR_VOL_THR)
ATR_VOL_THR   = 8.0   # batas ATR tinggi/rendah
MIN_ATR       = 2.0   # skip candle ATR terlalu kecil
MIN_ADX       = 20.0  # hanya trade saat trending (ADX > 20)
ZONE_PERIOD   = 50
ZONE_PCT      = 0.003
SEED          = 42
np.random.seed(SEED)
print("Config OK — LOOKAHEAD=20  SL=6pip  TP=8pip  ADX>20")


Config OK — LOOKAHEAD=20  SL=6pip  TP=8pip  ADX>20


## 1. Load Data

In [3]:
def load_mt5(path):
    df = pd.read_csv(path, sep="	")
    df.columns = [c.strip("<>").lower() for c in df.columns]
    if "date" in df.columns and "time" in df.columns:
        df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"])
        df = df.set_index("datetime")
    elif "date" in df.columns:
        df.index = pd.to_datetime(df["date"])
    if "tickvol" in df.columns and "volume" not in df.columns:
        df = df.rename(columns={"tickvol": "volume"})
    df = df[["open","high","low","close","volume"]].dropna()
    df = df[~df.index.duplicated(keep="last")].sort_index()
    return df

# Load M5 utama
BASE = Path("C:/Users/muham/Desktop/trader-ai")
M5_PATH = BASE / "ai/ml/data/XAUUSDm_M5.csv"
H1_PATH = BASE / "ai/ml/data/XAUUSDm_H1.csv"

df_raw = load_mt5(M5_PATH)
print(f"M5: {len(df_raw):,} candles | {df_raw.index[0]} -> {df_raw.index[-1]}")

# Load H1 untuk trend konteks
df_h1 = None
if H1_PATH.exists():
    df_h1 = load_mt5(H1_PATH)
    df_h1["h1_ema50"]  = df_h1["close"].ewm(span=50,  adjust=False).mean()
    df_h1["h1_ema200"] = df_h1["close"].ewm(span=200, adjust=False).mean()
    df_h1["h1_bull"]   = (df_h1["h1_ema50"] > df_h1["h1_ema200"]).astype(int)
    df_h1["h1_rsi"]    = 100 - 100/(1 + df_h1["close"].diff().clip(lower=0).ewm(span=14).mean() /
                          (-df_h1["close"].diff().clip(upper=0).ewm(span=14).mean() + 1e-9))
    df_h1["h1_adx"]    = df_h1["close"].diff().abs().ewm(span=14, adjust=False).mean()
    print(f"H1: {len(df_h1):,} candles | {df_h1.index[0]} -> {df_h1.index[-1]}")
else:
    print("H1 tidak ditemukan — training tanpa HTF context")

df_raw.tail(3)


M5: 100,119 candles | 2024-10-29 17:15:00 -> 2026-03-31 17:15:00
H1: 5,989 candles | 2025-04-01 00:00:00 -> 2026-04-07 07:00:00


,open,high,low,close,volume
datetime,,,,,
2026-03-31 17:05:00,4647.901,4649.436,4636.916,4639.255,3414
2026-03-31 17:10:00,4639.171,4645.336,4636.421,4642.992,3534
2026-03-31 17:15:00,4642.934,4644.997,4641.838,4644.044,706


## 2. Feature Engineering

In [4]:
def add_features(df):
    d=df.copy(); c=d["close"]; h=d["high"]; l=d["low"]; o=d["open"]
    tr=pd.concat([h-l,(h-c.shift()).abs(),(l-c.shift()).abs()],axis=1).max(axis=1)
    d["atr"]=tr.ewm(span=14,adjust=False).mean()
    d["atr5"]=tr.ewm(span=5,adjust=False).mean()
    d["atr_norm"]=d["atr5"]/(d["atr"]+1e-9)
    d["body"]=(c-o).abs(); d["rng"]=h-l
    d["upper_wick"]=h-pd.concat([c,o],axis=1).max(axis=1)
    d["lower_wick"]=pd.concat([c,o],axis=1).min(axis=1)-l
    d["body_pct"]=d["body"]/(d["rng"]+1e-9)
    d["is_bull"]=(c>o).astype(int); d["range_atr"]=d["rng"]/(d["atr"]+1e-9)
    # Fix4 Candle patterns
    d["bullish_engulf"]=((c>o)&(c.shift(1)<o.shift(1))&(c>o.shift(1))&(o<c.shift(1))).astype(int)
    d["bearish_engulf"]=((c<o)&(c.shift(1)>o.shift(1))&(c<o.shift(1))&(o>c.shift(1))).astype(int)
    d["doji"]=(d["body"]<d["rng"]*0.1).astype(int)
    d["hammer"]=((d["lower_wick"]>2*d["body"])&(d["upper_wick"]<d["body"])).astype(int)
    d["shooting_star"]=((d["upper_wick"]>2*d["body"])&(d["lower_wick"]<d["body"])).astype(int)
    d["pin_bar_bull"]=(d["lower_wick"]>d["rng"]*0.6).astype(int)
    d["pin_bar_bear"]=(d["upper_wick"]>d["rng"]*0.6).astype(int)
    d["inside_bar"]=((h<h.shift(1))&(l>l.shift(1))).astype(int)
    # EMAs
    for sp in [9,20,50,200]: d[f"ema{sp}"]=c.ewm(span=sp,adjust=False).mean()
    d["price_ema9"]=(c-d["ema9"])/(d["atr"]+1e-9)
    d["price_ema20"]=(c-d["ema20"])/(d["atr"]+1e-9)
    d["price_ema50"]=(c-d["ema50"])/(d["atr"]+1e-9)
    d["ema9_20"]=(d["ema9"]-d["ema20"])/(d["atr"]+1e-9)
    d["ema20_50"]=(d["ema20"]-d["ema50"])/(d["atr"]+1e-9)
    d["trend_ema20"]=(c>d["ema20"]).astype(int)
    d["trend_ema50"]=(c>d["ema50"]).astype(int)
    d["trend_ema200"]=(c>d["ema200"]).astype(int)
    # RSI
    delta=c.diff(); gain=delta.clip(lower=0).rolling(14).mean()
    loss=(-delta.clip(upper=0)).rolling(14).mean()
    d["rsi"]=100-100/(1+gain/(loss+1e-9))
    d["rsi_lag1"]=d["rsi"].shift(1); d["rsi_slope"]=d["rsi"]-d["rsi_lag1"]
    d["rsi_ob"]=(d["rsi"]>70).astype(int); d["rsi_os"]=(d["rsi"]<30).astype(int)
    # MACD
    ema12=c.ewm(span=12).mean(); ema26=c.ewm(span=26).mean()
    d["macd"]=(ema12-ema26)/(d["atr"]+1e-9); d["macd_sig"]=d["macd"].ewm(span=9).mean()
    d["macd_hist"]=d["macd"]-d["macd_sig"]
    d["macd_cross_up"]=((d["macd_hist"]>0)&(d["macd_hist"].shift(1)<=0)).astype(int)
    d["macd_cross_dn"]=((d["macd_hist"]<0)&(d["macd_hist"].shift(1)>=0)).astype(int)
    # ADX
    dm_p=(h-h.shift()).clip(lower=0); dm_n=(l.shift()-l).clip(lower=0)
    di_p=100*dm_p.ewm(span=14).mean()/(d["atr"]+1e-9)
    di_n=100*dm_n.ewm(span=14).mean()/(d["atr"]+1e-9)
    dx=100*(di_p-di_n).abs()/(di_p+di_n+1e-9)
    d["adx"]=dx.ewm(span=14).mean(); d["di_diff"]=(di_p-di_n)/(di_p+di_n+1e-9)
    d["trending"]=(d["adx"]>25).astype(int)
    bb_m=c.rolling(20).mean(); bb_s=c.rolling(20).std()
    d["bb_width"]=4*bb_s/(bb_m+1e-9); d["bb_pos"]=(c-(bb_m-2*bb_s))/(4*bb_s+1e-9)
    d["bb_squeeze"]=(4*bb_s<(4*bb_s).rolling(20).mean()).astype(int)
    lo14=l.rolling(14).min(); hi14=h.rolling(14).max()
    d["stoch_k"]=100*(c-lo14)/(hi14-lo14+1e-9)
    d["stoch_d"]=d["stoch_k"].rolling(3).mean()
    d["vol_ma20"]=d["volume"].rolling(20).mean()
    d["vol_ratio"]=d["volume"]/(d["vol_ma20"]+1e-9)
    d["vol_spike"]=(d["vol_ratio"]>1.5).astype(int)
    tp_m=(h+l+c)/3; mf=tp_m*d["volume"]
    pm=mf.where(tp_m>tp_m.shift(1),0).rolling(14).sum()
    nm=mf.where(tp_m<tp_m.shift(1),0).rolling(14).sum()
    d["mfi"]=100-100/(1+pm/(nm+1e-9))
    for lag in [1,2,3,5]: d[f"ret{lag}"]=c.pct_change(lag)
    d["mom5"]=c/c.shift(5)-1; d["mom10"]=c/c.shift(10)-1
    # Fix2 Market structure
    d["hh"]=(h>h.shift(1)).astype(int); d["ll"]=(l<l.shift(1)).astype(int)
    d["hl"]=(l>l.shift(1)).astype(int); d["lh"]=(h<h.shift(1)).astype(int)
    d["hh2"]=((h>h.shift(1))&(h.shift(1)>h.shift(2))).astype(int)
    d["ll2"]=((l<l.shift(1))&(l.shift(1)<l.shift(2))).astype(int)
    d["swing_high"]=((h>h.shift(1))&(h>h.shift(-1))).astype(int)
    d["swing_low"]=((l<l.shift(1))&(l<l.shift(-1))).astype(int)
    rh=h.rolling(10).max(); rl=l.rolling(10).min()
    d["bos_bull"]=((c>rh.shift(1))&(c.shift(1)<=rh.shift(1))).astype(int)
    d["bos_bear"]=((c<rl.shift(1))&(c.shift(1)>=rl.shift(1))).astype(int)
    d["struct_bull"]=d["hh"]+d["hl"]-d["ll"]-d["lh"]
    # Fix3 Liquidity
    eq_thr=d["atr"]*0.1
    d["equal_high"]=(abs(h-h.shift(1))<eq_thr).astype(int)
    d["equal_low"]=(abs(l-l.shift(1))<eq_thr).astype(int)
    d["liq_sweep_up"]=((h>h.shift(1))&(c<h.shift(1))).astype(int)
    d["liq_sweep_dn"]=((l<l.shift(1))&(c>l.shift(1))).astype(int)
    d["stop_hunt_bull"]=((l<l.rolling(5).min().shift(1))&(c>l.rolling(5).min().shift(1))).astype(int)
    d["stop_hunt_bear"]=((h>h.rolling(5).max().shift(1))&(c<h.rolling(5).max().shift(1))).astype(int)
    # Fix5 Area context
    d["r50"]=h.rolling(ZONE_PERIOD).max(); d["s50"]=l.rolling(ZONE_PERIOD).min()
    zthr=c*ZONE_PCT
    d["near_resistance"]=(abs(c-d["r50"])<zthr).astype(int)
    d["near_support"]=(abs(c-d["s50"])<zthr).astype(int)
    d["dist_res"]=(d["r50"]-c)/(d["atr"]+1e-9)
    d["dist_sup"]=(c-d["s50"])/(d["atr"]+1e-9)
    d["range_pos"]=(c-d["s50"])/(d["r50"]-d["s50"]+1e-9)
    # Fix7 Session
    try: hour=pd.to_datetime(d.index).hour
    except: hour=pd.Series(0,index=d.index)
    d["hour"]=hour
    d["is_london"]=((hour>=7)&(hour<16)).astype(int)
    d["is_newyork"]=((hour>=12)&(hour<21)).astype(int)
    d["is_overlap"]=((hour>=12)&(hour<16)).astype(int)
    d["in_session"]=((d["is_london"]==1)|(d["is_newyork"]==1)).astype(int)
    for col in ["rsi","adx","macd_hist","body_pct","struct_bull"]:
        d[f"{col}_lag1"]=d[col].shift(1); d[f"{col}_lag2"]=d[col].shift(2)

    # H1 Trend Context (HTF bias)
    try:
        if df_h1 is not None:
            _h1 = df_h1[["h1_bull","h1_rsi"]].copy()
            _h1 = _h1.reindex(d.index, method="ffill")
            d["h1_bull"]  = _h1["h1_bull"].fillna(0).astype(int)
            d["h1_rsi"]   = _h1["h1_rsi"].fillna(50.0)
            d["h1_align"] = ((d["h1_bull"]==1)&(d["is_bull"]==1)).astype(int) - ((d["h1_bull"]==0)&(d["is_bull"]==0)).astype(int)
        else:
            d["h1_bull"]=0; d["h1_rsi"]=50.0; d["h1_align"]=0
    except Exception:
        d["h1_bull"]=0; d["h1_rsi"]=50.0; d["h1_align"]=0
    return d

df=add_features(df_raw)
print(f"Features: {len(df.columns)} | Rows: {len(df):,}")

Features: 104 | Rows: 100,119


## 3. ATR-Based Labels (Fix #1)

In [5]:
def create_labels(df, lookahead=LOOKAHEAD):
    """Fixed-pip label: BUY(1) SELL(0) resolved by SL/TP hit."""
    d=df.copy()
    closes=d["close"].values; highs=d["high"].values; lows=d["low"].values
    labels=[]
    for i in range(len(d)-lookahead):
        entry=closes[i]
        # Fixed pip — sama persis dengan live bot
        tp_b=entry+FIXED_TP; sl_b=entry-FIXED_SL
        tp_s=entry-FIXED_TP; sl_s=entry+FIXED_SL
        buy_win=False; sell_win=False
        buy_done=False; sell_done=False
        for j in range(1, lookahead+1):
            k=i+j
            if k>=len(d): break
            fh=highs[k]; fl=lows[k]
            if not buy_done:
                if fh>=tp_b:  buy_win=True;  buy_done=True
                elif fl<=sl_b: buy_done=True
            if not sell_done:
                if fl<=tp_s:  sell_win=True; sell_done=True
                elif fh>=sl_s: sell_done=True
            if buy_done and sell_done: break
        if buy_win and not sell_win:    labels.append(1)
        elif sell_win and not buy_win:  labels.append(0)
        else: labels.append(None)  # ambiguous — skip
    labels+=[None]*lookahead
    d["label"]=labels
    return d

df=create_labels(add_features(df_raw))
v=df["label"].dropna()
n_buy=int((v==1).sum()); n_sell=int((v==0).sum())
print(f"Labels: {len(v):,} valid | BUY:{n_buy}({n_buy/len(v)*100:.1f}%) SELL:{n_sell}({n_sell/len(v)*100:.1f}%)")
print(f"SL={FIXED_SL}pip TP={FIXED_TP}pip LOOKAHEAD={LOOKAHEAD} candles ({LOOKAHEAD*5}min)")


Labels: 55,842 valid | BUY:28220(50.5%) SELL:27622(49.5%)
SL=6.0pip TP=8.0pip LOOKAHEAD=20 candles (100min)


## 4. Prepare Dataset

In [6]:
EXCLUDE=["label","open","high","low","close","volume",
         "ema9","ema20","ema50","ema200","r50","s50","rng"]
feat_cols=[col for col in df.columns if col not in EXCLUDE
           and df[col].dtype in [np.float64,np.int64,float,int]]

df_c=df.dropna(subset=["label"]+feat_cols).copy()
df_c["label"]=df_c["label"].astype(int)
X=df_c[feat_cols].ffill().fillna(0).replace([np.inf,-np.inf],0)
y=df_c["label"]

# Check label distribution
n_sell=int((y==0).sum()); n_buy=int((y==1).sum())
print(f"Label dist => SELL(0):{n_sell}  BUY(1):{n_buy}  ratio:{n_sell/(n_buy+1e-9):.1f}:1")
if n_buy < 50:
    raise ValueError(f"Terlalu sedikit label BUY ({n_buy}). Perbesar data history.")

split=int(len(X)*0.8)
X_tr,X_te=X.iloc[:split],X.iloc[split:]
y_tr,y_te=y.iloc[:split],y.iloc[split:]
spw=float((y_tr==0).sum()/(y_tr==1).sum()) if (y_tr==1).sum()>0 else 1.0
print(f"Train:{len(X_tr):,}  Test:{len(X_te):,}  spw={spw:.2f}")

scaler=RobustScaler()
X_tr_s=scaler.fit_transform(X_tr); X_te_s=scaler.transform(X_te)
sel_m=lgb.LGBMClassifier(n_estimators=200,random_state=SEED,verbose=-1,class_weight="balanced")
sel_m.fit(X_tr_s,y_tr)
selector=SelectFromModel(sel_m,prefit=True,threshold="mean")
X_tr_sel=selector.transform(X_tr_s); X_te_sel=selector.transform(X_te_s)
sel_feats=[feat_cols[i] for i in selector.get_support(indices=True)]
print(f"Features:{len(feat_cols)}->{len(sel_feats)}")


Label dist => SELL(0):27622  BUY(1):28220  ratio:1.0:1
Train:44,673  Test:11,169  spw=0.95


Features:91->30


## 5. Optuna Tuning

In [7]:
tscv=TimeSeriesSplit(n_splits=5)

# Sanity check before tuning
assert not np.isnan(X_tr_sel).any(), "X_tr_sel has NaN!"
assert not np.isinf(X_tr_sel).any(), "X_tr_sel has inf!"
vc = dict(zip(*np.unique(y_tr, return_counts=True)))
print(f"Data OK: X_tr_sel {X_tr_sel.shape}  labels={vc}")
# Check per-fold class presence
for fold,(tr_idx,va_idx) in enumerate(tscv.split(X_tr_sel)):
    uniq = np.unique(y_tr.iloc[va_idx])
    if len(uniq)<2: print(f"  WARNING fold {fold}: only class {uniq} in val")

def safe_cv(model):
    """Return mean AUC or 0.5 if any fold is single-class / error."""
    try:
        scores = cross_val_score(model, X_tr_sel, y_tr, cv=tscv,
                                  scoring="roc_auc", n_jobs=1, error_score=np.nan)
        valid = scores[~np.isnan(scores)]
        return float(valid.mean()) if len(valid) else 0.5
    except Exception:
        return 0.5

def obj_lgb(trial):
    p=dict(
        n_estimators=trial.suggest_int("n",100,300),
        max_depth=trial.suggest_int("d",4,9),
        num_leaves=trial.suggest_int("nl",20,70),
        learning_rate=trial.suggest_float("lr",0.01,0.15,log=True),
        min_child_samples=trial.suggest_int("mc",20,100),
        subsample=trial.suggest_float("ss",0.6,1.0),
        colsample_bytree=trial.suggest_float("cs",0.6,1.0),
        reg_alpha=trial.suggest_float("ra",1e-4,1.0,log=True),
        reg_lambda=trial.suggest_float("rl",1e-4,1.0,log=True),
        class_weight="balanced",random_state=SEED,verbose=-1)
    return safe_cv(lgb.LGBMClassifier(**p))

study_lgb=optuna.create_study(direction="maximize")
study_lgb.optimize(obj_lgb,n_trials=20,timeout=120,show_progress_bar=True)

good_lgb=[t for t in study_lgb.trials if t.value and t.value>0.5]
if not good_lgb:
    print("Semua trial LGB NaN/default â€” pakai default params")
    best_lgb_p={"n":300,"d":6,"nl":40,"lr":0.05,"mc":50,"ss":0.8,"cs":0.8,"ra":0.01,"rl":0.01}
else:
    best_lgb_p=study_lgb.best_params
    print(f"LGB best AUC: {study_lgb.best_value:.4f}")

def obj_xgb(trial):
    spw = float((y_tr==0).sum()/(y_tr==1).sum()) if (y_tr==1).sum()>0 else 1.0
    p=dict(
        n_estimators=trial.suggest_int("n",100,300),
        max_depth=trial.suggest_int("d",3,8),
        learning_rate=trial.suggest_float("lr",0.01,0.15,log=True),
        subsample=trial.suggest_float("ss",0.6,1.0),
        colsample_bytree=trial.suggest_float("cs",0.6,1.0),
        reg_alpha=trial.suggest_float("ra",1e-4,1.0,log=True),
        reg_lambda=trial.suggest_float("rl",1e-4,1.0,log=True),
        scale_pos_weight=spw,
        random_state=SEED,verbosity=0,eval_metric="auc")
    return safe_cv(xgb.XGBClassifier(**p))

study_xgb=optuna.create_study(direction="maximize")
study_xgb.optimize(obj_xgb,n_trials=15,timeout=90,show_progress_bar=True)

good_xgb=[t for t in study_xgb.trials if t.value and t.value>0.5]
if not good_xgb:
    print("Semua trial XGB NaN/default â€” pakai default params")
    best_xgb_p={"n":300,"d":5,"lr":0.05,"ss":0.8,"cs":0.8,"ra":0.01,"rl":0.01}
else:
    best_xgb_p=study_xgb.best_params
    print(f"XGB best AUC: {study_xgb.best_value:.4f}")


Data OK: X_tr_sel (44673, 30)  labels={np.int64(0): np.int64(21767), np.int64(1): np.int64(22906)}


  0%|          | 0/20 [00:00<?, ?it/s]

LGB best AUC: 0.6855


  0%|          | 0/15 [00:00<?, ?it/s]

XGB best AUC: 0.6796


## 6. Train Stacking Ensemble

In [8]:
import sys as _sys, os as _os
_root = str(_os.path.abspath(_os.path.join(_os.getcwd(), "../../..")))
if _root not in _sys.path: _sys.path.insert(0, _root)

def mk_lgb(p):
    km={"n":"n_estimators","d":"max_depth","nl":"num_leaves","lr":"learning_rate",
        "mc":"min_child_samples","ss":"subsample","cs":"colsample_bytree","ra":"reg_alpha","rl":"reg_lambda"}
    return {km.get(k,k):v for k,v in p.items()}

def mk_xgb(p):
    km={"n":"n_estimators","d":"max_depth","lr":"learning_rate","ss":"subsample",
        "cs":"colsample_bytree","ra":"reg_alpha","rl":"reg_lambda"}
    return {km.get(k,k):v for k,v in p.items()}

# EnsembleModel diimport dari ai.ml.predictor agar pickle path benar
from ai.ml.predictor import EnsembleModel

lgb_m=lgb.LGBMClassifier(**mk_lgb(best_lgb_p),class_weight="balanced",random_state=SEED,verbose=-1)
xgb_m=xgb.XGBClassifier(**mk_xgb(best_xgb_p),scale_pos_weight=spw,
                         random_state=SEED,verbosity=0,eval_metric="auc")
et_m=ExtraTreesClassifier(n_estimators=300,max_depth=8,class_weight="balanced",
                          random_state=SEED,n_jobs=-1)
rf_m=RandomForestClassifier(n_estimators=200,max_depth=8,class_weight="balanced",
                            random_state=SEED,n_jobs=-1)

model=EnsembleModel([
    ("lgb",lgb_m),("xgb",xgb_m),("et",et_m),("rf",rf_m)
])
model.fit(X_tr_sel,y_tr)
print(f"Model trained OK  classes={model.classes_}")


Model trained OK  classes=[0 1]


## 7. Evaluasi + Threshold Optimization

In [9]:
proba_te=model.predict_proba(X_te_sel)
prob_buy=proba_te[:,1]; prob_sell=proba_te[:,0]
auc=roc_auc_score(y_te,prob_buy)
print(f"Test AUC: {auc:.4f}")

best_thresh,best_f1=0.55,0.0
for thr in [x/100 for x in range(50,80)]:
    pred=(prob_buy>=thr).astype(int)
    if pred.sum()<10: continue
    f=f1_score(y_te,pred,zero_division=0)
    if f>best_f1: best_f1=f; best_thresh=thr

print(f"Optimal threshold: {best_thresh:.2f}  F1={best_f1:.3f}")
print(classification_report(y_te,(prob_buy>=best_thresh).astype(int),target_names=["SELL","BUY"]))


Test AUC: 0.6757


Optimal threshold: 0.50  F1=0.617
              precision    recall  f1-score   support

        SELL       0.65      0.63      0.64      5855
         BUY       0.61      0.63      0.62      5314

    accuracy                           0.63     11169
   macro avg       0.63      0.63      0.63     11169
weighted avg       0.63      0.63      0.63     11169



## 8. Walk-Forward Validation

In [10]:
wf_results=[]
for fold,(tr_i,te_i) in enumerate(tscv.split(X)):
    y_val=y.iloc[te_i]
    if len(np.unique(y_val))<2:
        print(f"  Fold {fold+1}: SKIP â€” single class in val")
        continue
    try:
        m=lgb.LGBMClassifier(**mk_lgb(best_lgb_p),class_weight="balanced",random_state=SEED,verbose=-1)
        Xt=selector.transform(scaler.transform(X.iloc[tr_i]))
        Xv=selector.transform(scaler.transform(X.iloc[te_i]))
        m.fit(Xt,y.iloc[tr_i])
        p=m.predict_proba(Xv)[:,1]
        a=roc_auc_score(y_val,p)
        pr=precision_score(y_val,(p>=best_thresh).astype(int),zero_division=0)
        wf_results.append({"fold":fold+1,"auc":a,"precision":pr})
        print(f"  Fold {fold+1}: AUC={a:.4f}  Prec={pr:.3f}")
    except Exception as e:
        print(f"  Fold {fold+1}: ERROR â€” {e}")

if wf_results:
    wf_df=pd.DataFrame(wf_results)
    wf_auc_mean=wf_df["auc"].mean(); wf_auc_std=wf_df["auc"].std()
    print(f"WF AUC: {wf_auc_mean:.4f} +/- {wf_auc_std:.4f}")
else:
    print("Semua fold gagal â€” set WF AUC default")
    wf_auc_mean=0.5; wf_auc_std=0.0


  Fold 1: AUC=0.5750  Prec=0.586


  Fold 2: AUC=0.7298  Prec=0.675


  Fold 3: AUC=0.7396  Prec=0.669


  Fold 4: AUC=0.7062  Prec=0.657


  Fold 5: AUC=0.6839  Prec=0.610
WF AUC: 0.6869 +/- 0.0662


## 9. Hybrid Backtest (Fix #6 #7 #8 #9)

In [11]:
INITIAL_BALANCE = 100.0

def backtest_hybrid(df_feat, model, scaler, selector, feat_cols,
                    start_idx=0, initial_balance=INITIAL_BALANCE):
    trades=[]; balance=initial_balance
    rows=df_feat.iloc[start_idx:].copy()
    X_all=rows[feat_cols].ffill().fillna(0).replace([np.inf,-np.inf],0)
    proba=model.predict_proba(selector.transform(scaler.transform(X_all)))
    for i,(idx,row) in enumerate(rows.iterrows()):
        pb=float(proba[i,1]); ps=float(proba[i,0])
        atr=float(row.get("atr",0)); close=float(row["close"])
        adx=float(row.get("adx",0))
        # Filter: ATR minimum + ADX trending
        if atr < MIN_ATR: continue
        if adx < MIN_ADX: continue  # skip sideways
        if int(row.get("in_session",1))==0: continue
        # Dynamic threshold
        thr=PROB_HIGH_VOL if atr>=ATR_VOL_THR else PROB_LOW_VOL
        if pb>=thr:   direction="BUY"
        elif ps>=thr: direction="SELL"
        else: continue
        # H1 trend filter — hanya trade searah H1
        h1_bull=int(row.get("h1_bull",1))
        if direction=="BUY"  and h1_bull==0: continue  # M5 BUY tapi H1 bearish
        if direction=="SELL" and h1_bull==1: continue  # M5 SELL tapi H1 bullish
        # Area filter
        if direction=="BUY"  and int(row.get("near_resistance",0))==1: continue
        if direction=="SELL" and int(row.get("near_support",0))==1: continue
        # FIXED pip SL/TP — sama persis dengan live bot
        sl_d=FIXED_SL; tp_d=FIXED_TP
        entry=close
        tp=entry+tp_d if direction=="BUY" else entry-tp_d
        sl=entry-sl_d if direction=="BUY" else entry+sl_d
        result="TIMEOUT"; pnl_pts=0.0
        future=rows.iloc[i+1:i+1+LOOKAHEAD]
        for _,fr in future.iterrows():
            fh=float(fr["high"]); fl=float(fr["low"])
            if direction=="BUY":
                if fh>=tp: result="WIN";  pnl_pts=tp_d;  break
                if fl<=sl: result="LOSS"; pnl_pts=-sl_d; break
            else:
                if fl<=tp: result="WIN";  pnl_pts=tp_d;  break
                if fh>=sl: result="LOSS"; pnl_pts=-sl_d; break
        if result=="TIMEOUT": continue  # skip — tidak kena SL/TP dalam LOOKAHEAD
        pnl_usd=pnl_pts*LOT*100-SPREAD_USD
        balance+=pnl_usd
        trades.append({"time":idx,"direction":direction,"entry":round(entry,2),
                       "tp":round(tp,2),"sl":round(sl,2),"result":result,
                       "pnl_pts":round(pnl_pts,3),"pnl_usd":round(pnl_usd,2),
                       "balance":round(balance,2),"prob":round(max(pb,ps),4),
                       "atr":round(atr,3),"adx":round(adx,1)})
    return pd.DataFrame(trades)

trades_df=backtest_hybrid(df_c, model, scaler, selector, feat_cols, start_idx=split)

if len(trades_df)==0:
    print("Tidak ada trade — turunkan threshold atau periksa ADX filter")
else:
    wins=int((trades_df["result"]=="WIN").sum())
    total=len(trades_df); wr=wins/total*100
    pnl=trades_df["pnl_usd"].sum()
    avg_w=trades_df[trades_df["result"]=="WIN"]["pnl_usd"].mean()
    avg_l=trades_df[trades_df["result"]=="LOSS"]["pnl_usd"].mean()
    rr=abs(avg_w/avg_l) if avg_l else 0
    nb_=int((trades_df["direction"]=="BUY").sum())
    ns_=int((trades_df["direction"]=="SELL").sum())
    final_bal=INITIAL_BALANCE+pnl; pct=pnl/INITIAL_BALANCE*100
    print("="*60)
    print(f"  BACKTEST  (SL={FIXED_SL}pip TP={FIXED_TP}pip ADX>{MIN_ADX} LOOK={LOOKAHEAD}c)")
    print("="*60)
    print(f"  Total    : {total}  |  BUY:{nb_}  SELL:{ns_}")
    print(f"  WIN      : {wins} ({wr:.1f}%)")
    print(f"  LOSS     : {total-wins} ({100-wr:.1f}%)")
    print(f"  Avg WIN  : ${avg_w:.2f}  |  Avg LOSS: ${avg_l:.2f}")
    print(f"  RR       : 1:{rr:.2f}")
    print(f"  P&L      : ${pnl:.2f}  ({pct:+.1f}%)")
    print(f"  Saldo Akhir: ${final_bal:.2f}")
    print("="*60)
    dcols=["time","direction","entry","result","pnl_usd","balance","prob","adx"]
    print(trades_df[dcols].tail(15).to_string(index=False))


  BACKTEST  (SL=6.0pip TP=8.0pip ADX>20.0 LOOK=20c)
  Total    : 1420  |  BUY:657  SELL:763
  WIN      : 927 (65.3%)
  LOSS     : 493 (34.7%)
  Avg WIN  : $7.75  |  Avg LOSS: $-6.25
  RR       : 1:1.24
  P&L      : $4103.00  (+4103.0%)
  Saldo Akhir: $4203.00
               time direction   entry result  pnl_usd  balance   prob  adx
2026-03-31 09:20:00      SELL 4567.45    WIN     7.75  4122.50 0.6091 26.5
2026-03-31 09:25:00      SELL 4562.30    WIN     7.75  4130.25 0.6159 29.5
2026-03-31 09:30:00      SELL 4557.07    WIN     7.75  4138.00 0.5803 34.9
2026-03-31 09:45:00      SELL 4558.57    WIN     7.75  4145.75 0.5569 44.4
2026-03-31 10:00:00      SELL 4560.14    WIN     7.75  4153.50 0.6958 36.5
2026-03-31 10:05:00      SELL 4560.34    WIN     7.75  4161.25 0.6991 32.8
2026-03-31 10:10:00      SELL 4557.06    WIN     7.75  4169.00 0.6897 30.5
2026-03-31 10:15:00      SELL 4558.51    WIN     7.75  4176.75 0.7005 29.0
2026-03-31 10:20:00      SELL 4558.49    WIN     7.75  4184.50 0.

## 10. Charts

In [12]:
if len(trades_df) > 0:
    import matplotlib
    matplotlib.use('Agg')
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Equity / Balance Curve
    ax = axes[0, 0]
    ax.plot(trades_df["balance"], color="green", linewidth=1.5)
    ax.axhline(INITIAL_BALANCE, color="gray", linestyle="--", alpha=0.5)
    ax.fill_between(range(len(trades_df)), trades_df["balance"], INITIAL_BALANCE,
                    where=trades_df["balance"] >= INITIAL_BALANCE, alpha=0.2, color="green")
    ax.fill_between(range(len(trades_df)), trades_df["balance"], INITIAL_BALANCE,
                    where=trades_df["balance"] < INITIAL_BALANCE, alpha=0.2, color="red")
    pnl_total = trades_df["pnl_usd"].sum()
    ax.set_title(f"Balance Curve  Start=${INITIAL_BALANCE:.0f}  Final=${INITIAL_BALANCE+pnl_total:.2f}  P&L=${pnl_total:.2f}")

    # P&L per trade
    ax = axes[0, 1]
    colors = ["green" if r == "WIN" else "red" for r in trades_df["result"]]
    ax.bar(range(len(trades_df)), trades_df["pnl_usd"], color=colors, alpha=0.7)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title("P&L per Trade")

    # BUY vs SELL breakdown
    ax = axes[1, 0]
    bw  = len(trades_df[(trades_df["direction"]=="BUY")  & (trades_df["result"]=="WIN")])
    bl  = len(trades_df[(trades_df["direction"]=="BUY")  & (trades_df["result"]=="LOSS")])
    sw  = len(trades_df[(trades_df["direction"]=="SELL") & (trades_df["result"]=="WIN")])
    sl_ = len(trades_df[(trades_df["direction"]=="SELL") & (trades_df["result"]=="LOSS")])
    w = 0.35
    ax.bar([0-w/2, 1-w/2], [bw, sw], width=w, label="WIN",  color="green", alpha=0.8)
    ax.bar([0+w/2, 1+w/2], [bl, sl_], width=w, label="LOSS", color="red",   alpha=0.8)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["BUY", "SELL"]); ax.legend()
    ax.set_title("BUY vs SELL")

    # Probability distribution
    ax = axes[1, 1]
    ax.hist(trades_df[trades_df["result"]=="WIN"]["prob"],  bins=20, alpha=0.6, label="WIN",  color="green")
    ax.hist(trades_df[trades_df["result"]=="LOSS"]["prob"], bins=20, alpha=0.6, label="LOSS", color="red")
    ax.set_title("Probability Distribution"); ax.legend()

    plt.tight_layout()
    os.makedirs("../results", exist_ok=True)
    plt.savefig("../results/backtest_hybrid.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Chart saved: results/backtest_hybrid.png")


Chart saved: results/backtest_hybrid.png


## 11. Simpan Model

In [13]:
os.makedirs("../models",  exist_ok=True)
os.makedirs("../results", exist_ok=True)

bundle = {
    "model"         : model,
    "scaler"        : scaler,
    "selector"      : selector,
    "feature_cols"  : feat_cols,
    "selected_feats": sel_feats,
    "prob_threshold": best_thresh,
    "tp_mult"       : ATR_TP_MULT,
    "sl_mult"       : ATR_SL_MULT,
    "symbol"        : SYMBOL,
    "created_at"    : datetime.now().isoformat(),
    "auc_test"      : float(auc),
    "wf_auc_mean"   : float(wf_auc_mean),
}
joblib.dump(bundle, "../models/xauusd_scalping_model.joblib", compress=3)

meta = {k: v for k, v in bundle.items() if k not in ("model","scaler","selector")}
if len(trades_df) > 0:
    wins_s = int((trades_df["result"]=="WIN").sum())
    total_s = len(trades_df)
    pnl_s   = trades_df["pnl_usd"].sum()
    meta["backtest_initial_balance"] = INITIAL_BALANCE
    meta["backtest_final_balance"]   = round(INITIAL_BALANCE + pnl_s, 2)
    meta["backtest_profit_usd"]      = round(pnl_s, 2)
    meta["backtest_profit_pct"]      = round(pnl_s / INITIAL_BALANCE * 100, 1)
    meta["backtest_winrate"]         = round(wins_s / total_s * 100, 1)
    meta["backtest_trades"]          = total_s
    trades_df.to_csv("../results/trades.csv", index=False)

with open("../models/xauusd_scalping_model_meta.json","w") as f:
    json.dump(meta, f, indent=2, default=str)

print("Model saved : xauusd_scalping_model.joblib")
print("Meta  saved : xauusd_scalping_model_meta.json")
print(f"Trades saved: results/trades.csv  ({len(trades_df)} trades)")
print()
print(f"AUC Test    : {auc:.4f}")
print(f"WF AUC Mean : {wf_auc_mean:.4f}")
if len(trades_df) > 0:
    pnl_s = trades_df['pnl_usd'].sum()
    wr_s  = (trades_df['result']=='WIN').mean() * 100
    final = INITIAL_BALANCE + pnl_s
    print(f"Win Rate    : {wr_s:.1f}%")
    print(f"Total P&L   : ${pnl_s:.2f}")
    print(f"Modal $100  -> ${final:.2f}  ({pnl_s/INITIAL_BALANCE*100:+.1f}%)")


Model saved : xauusd_scalping_model.joblib
Meta  saved : xauusd_scalping_model_meta.json
Trades saved: results/trades.csv  (1420 trades)

AUC Test    : 0.6757
WF AUC Mean : 0.6869
Win Rate    : 65.3%
Total P&L   : $4103.00
Modal $100  -> $4203.00  (+4103.0%)
